# 2교시. OCR 기반 텍스트 추출 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/02_ocr_basic.ipynb)

**이번 교시 행동:** 공개 한국 영수증에 실제 OCR을 실행하고 원본 위 좌표·신뢰도를 확인합니다.

**통과 증거:** `course_outputs/ocr_result.json`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. `TODO`가 있는 셀은 안내된 `None` 또는 짧은 값만 바꿉니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(current, total, title, action, expected, code_help):
    _show_learning_message(
        f"""---
### 🧪 실습 단계 {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# `OUTPUT_DIR`는 모든 산출물의 공통 폴더입니다. 업로드·다운로드 함수와 자료 로더를 등록하는 준비 셀이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 6, '공통 환경 준비', '결과 폴더와 실습 자료 다운로드 기능을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '`OUTPUT_DIR`는 모든 산출물의 공통 폴더입니다. 업로드·다운로드 함수와 자료 로더를 등록하는 준비 셀이므로 수정하지 않습니다.')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/document_ai_lecture_2026/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 6, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `USE_MY_RECEIPT=False`면 공개 영수증을 사용합니다. 내 영수증을 쓰려면 비식별 처리 후 이 값만 `True`로 바꿉니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 6, 'OCR 입력 한 장 준비', '공개 영수증과 장애 시 사용할 검수 데이터를 불러옵니다.', '요청 모드와 입력 파일명이 표시되어야 합니다.', '`USE_MY_RECEIPT=False`면 공개 영수증을 사용합니다. 내 영수증을 쓰려면 비식별 처리 후 이 값만 `True`로 바꿉니다.')

import io
from PIL import Image, ImageDraw

SAMPLE_IMAGE_PATH = (
    "sample_docs/public_receipts/korea/"
    "taebaek_restaurant_2025_redacted.png"
)
PREPARED_OCR_PATH = (
    "sample_docs/prepared/receipt_ocr_fallback.json"
)
lesson_assets = load_course_assets(
    SAMPLE_IMAGE_PATH,
    PREPARED_OCR_PATH,
)
receipt_image = Image.open(
    io.BytesIO(lesson_assets[SAMPLE_IMAGE_PATH])
).convert("RGB")
PREPARED_OCR_RESULT = json.loads(
    lesson_assets[PREPARED_OCR_PATH].decode("utf-8")
)
for item in PREPARED_OCR_RESULT:
    item["confidence_source"] = "not_available_prepared_fixture"
USE_MY_RECEIPT = False
INPUT_FILE_NAME = "taebaek_restaurant_2025_redacted.png"
if USE_MY_RECEIPT and not VALIDATION_MODE:
    from google.colab import files
    print(
        "카드·승인·전화·회원번호 등을 먼저 가린 "
        "영수증 이미지 한 장만 선택하세요."
    )
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("비식별 영수증 이미지 한 장만 선택하세요.")
    INPUT_FILE_NAME, uploaded_bytes = next(iter(uploaded.items()))
    receipt_image = Image.open(
        io.BytesIO(uploaded_bytes)
    ).convert("RGB")
RUN_LIVE_OCR = not VALIDATION_MODE
print("요청 모드:", "LIVE" if RUN_LIVE_OCR else "PREPARED_FALLBACK")
print("입력 파일:", INPUT_FILE_NAME)

complete_lab_step(2, 6, '요청 모드와 입력 파일명이 표시되어야 합니다.')


## 실행

기본값은 `LIVE`입니다. Colab에서 PaddleOCR 3.7과
`PP-OCRv5 Korean`을 사용합니다. PP-OCRv6가 최신 기본 계열이어도
한국어 전용 인식 모델은 PP-OCRv5 Korean을 사용합니다.

설치·모델 다운로드가 3분을 넘기면 중지합니다. 오류 메시지를 보존한
채 `PREPARED_FALLBACK`으로 전환하며, 전환 사실을 결과 JSON에 남깁니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `RUN_LIVE_OCR`이 실제 모델 실행 여부를 정합니다. `try`가 성공하면 `LIVE`, 실패하면 검수된
# `PREPARED_FALLBACK` 결과를 사용합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 6, 'PP-OCRv5 실행', '영수증 한 장을 LIVE OCR로 읽고 실패 시 복구 결과로 전환합니다.', '실행 모드·복구 사유·판독 영역 수를 확인합니다.', '`RUN_LIVE_OCR`이 실제 모델 실행 여부를 정합니다. `try`가 성공하면 `LIVE`, 실패하면 검수된 `PREPARED_FALLBACK` 결과를 사용합니다.')

OCR_RESULT = PREPARED_OCR_RESULT
SOURCE_MODE = "PREPARED_FALLBACK"
FALLBACK_REASON = "offline validator"

if RUN_LIVE_OCR:
    import subprocess
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q",
             "paddlepaddle==3.2.1", "paddleocr==3.7.0"]
        )
        from paddleocr import PaddleOCR

        image_path = OUTPUT_DIR / "golden_receipt.jpg"
        receipt_image.save(image_path)
        engine = PaddleOCR(
            lang="korean",
            ocr_version="PP-OCRv5",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            device="cpu",
        )
        page = list(engine.predict(str(image_path)))[0]
        payload = page.json() if callable(page.json) else page.json
        result = payload.get("res", payload)
        OCR_RESULT = [
            {
                "box": box,
                "text": text,
                "confidence": float(score),
            }
            for box, text, score in zip(
                result.get("rec_polys", []),
                result.get("rec_texts", []),
                result.get("rec_scores", []),
            )
        ]
        SOURCE_MODE = "LIVE"
        FALLBACK_REASON = ""
    except Exception as exc:
        SOURCE_MODE = "PREPARED_FALLBACK"
        FALLBACK_REASON = f"{type(exc).__name__}: {exc}"

print("실행 모드:", SOURCE_MODE)
if FALLBACK_REASON:
    print("복구 사유:", FALLBACK_REASON)
print("판독 영역:", len(OCR_RESULT))

complete_lab_step(3, 6, '실행 모드·복구 사유·판독 영역 수를 확인합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `OCR_RESULT`의 좌표를 이미지에 그리고 JSON으로 저장합니다. `scale_x`와 `scale_y`는 이미지 크기가 달라도 좌표를
# 맞춥니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 6, 'OCR 위치 시각화와 저장', '판독 영역을 원본 위에 그리고 JSON과 이미지를 저장합니다.', '`CHECKPOINT 1/1 PASS`와 두 결과 파일을 확인합니다.', '`OCR_RESULT`의 좌표를 이미지에 그리고 JSON으로 저장합니다. `scale_x`와 `scale_y`는 이미지 크기가 달라도 좌표를 맞춥니다.')

annotated = receipt_image.copy()
draw = ImageDraw.Draw(annotated)
scale_x = annotated.width / 900
scale_y = annotated.height / 1100
for item in OCR_RESULT:
    points = item["box"]
    xs = [point[0] * scale_x for point in points]
    ys = [point[1] * scale_y for point in points]
    draw.rectangle(
        (min(xs), min(ys), max(xs), max(ys)),
        outline="#0F766E",
        width=4,
    )
annotated_path = OUTPUT_DIR / "ocr_boxes.png"
annotated.save(annotated_path)

output = {
    "source_mode": SOURCE_MODE,
    "fallback_reason": FALLBACK_REASON,
    "input_file": INPUT_FILE_NAME,
    "items": [
        {**item, "matches_source": None, "review_note": ""}
        for item in OCR_RESULT
    ],
}
output_path = OUTPUT_DIR / "ocr_result.json"
output_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("CHECKPOINT 1/1 PASS:", SOURCE_MODE, output_path, annotated_path)

complete_lab_step(4, 6, '`CHECKPOINT 1/1 PASS`와 두 결과 파일을 확인합니다.')


## 내가 직접 채우는 3줄

금액·날짜처럼 원본 대조가 필요한 OCR 토큰을 키워드로 표시합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `review_keywords`의 세 `None`만 원본 대조에 사용할 문자열로 바꿉니다. OCR 원문에서 반드시 찾아야 할 값을 고르는
# 연습입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 6, '내 원본 대조 기준 입력', 'OCR 원문에서 반드시 확인할 키워드 세 개를 정합니다.', '빈칸 여부 또는 내가 입력한 키워드가 표시되어야 합니다.', '`review_keywords`의 세 `None`만 원본 대조에 사용할 문자열로 바꿉니다. OCR 원문에서 반드시 찾아야 할 값을 고르는 연습입니다.')

# TODO: 원본 대조할 키워드 세 개를 넣으세요.
review_keywords = [None, None, None]
if any(value is None for value in review_keywords):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(5, 6, '빈칸 여부 또는 내가 입력한 키워드가 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

날짜의 연도, 합계 라벨, 합계 금액처럼 영향이 큰 토큰을 고릅니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `ANSWER_REVIEW_KEYWORDS`는 공개 정답이고 `zipfile`은 JSON과 위치 이미지를 하나의 다운로드 파일로 묶습니다. 수정
# 없이 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 6, 'OCR 결과 묶음 완성', '공개 정답과 비교하고 OCR 산출물을 ZIP으로 묶습니다.', '대조 결과와 `lesson02_ocr_outputs.zip` 경로를 확인합니다.', '`ANSWER_REVIEW_KEYWORDS`는 공개 정답이고 `zipfile`은 JSON과 위치 이미지를 하나의 다운로드 파일로 묶습니다. 수정 없이 실행합니다.')

ANSWER_REVIEW_KEYWORDS = ["이태리", "2025", "76,000"]
marked = 0
for item in output["items"]:
    if any(
        keyword in item.get("text", "")
        for keyword in ANSWER_REVIEW_KEYWORDS
    ):
        item["review_note"] = "원본 대조 필수"
        marked += 1
output_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert marked == 3
print("전체 정답 · 원본 대조 표시:", marked, "개")
import zipfile
bundle_path = OUTPUT_DIR / "lesson02_ocr_outputs.zip"
with zipfile.ZipFile(bundle_path, "w") as archive:
    archive.write(output_path, output_path.name)
    archive.write(annotated_path, annotated_path.name)
print("한 번만 다운로드할 묶음:", bundle_path)
download_artifact(bundle_path)

complete_lab_step(6, 6, '대조 결과와 `lesson02_ocr_outputs.zip` 경로를 확인합니다.')
